# CutBoost

В этом ноутбуке я буду обучать CutBoost модель, которая будет ранжировать отобранных IALS кандидатов. 

## Импорт библиотек

In [10]:
import numpy as np
import pandas as pd
import scipy 
import os
import sys
import pickle
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import catboost

## Выгрузка данных

Выгрузка таблиц

In [11]:
clean_path = Path("../../../Tables/CleanTable")
items_dim = pd.read_csv(clean_path / "items_dim.csv")
items_stats = pd.read_csv(clean_path / "items_stats.csv")
users_clean = pd.read_csv(clean_path / "users_clean.csv")
user_item_year_clean = pd.read_csv(clean_path / "user_items_year_clean.csv")


In [21]:
items_dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 235 entries, 0 to 234
Data columns (total 3 columns):
 #   Column                                 Non-Null Count  Dtype
---  ------                                 --------------  -----
 0   id_item                                235 non-null    int64
 1   Название услуги или товара             235 non-null    str  
 2   Категория товара или услуги в продаже  235 non-null    str  
dtypes: int64(1), str(2)
memory usage: 5.6 KB


In [23]:
items_stats.info()

<class 'pandas.DataFrame'>
RangeIndex: 235 entries, 0 to 234
Data columns (total 9 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   id_item                                235 non-null    int64  
 1   Название услуги или товара             235 non-null    str    
 2   Категория товара или услуги в продаже  235 non-null    str    
 3   Количество оказанных услуг             235 non-null    int64  
 4   Доля от оказанных услуг в %            235 non-null    float64
 5   Выручка от продажи услуг, руб          235 non-null    float64
 6   % от общей выручки за услуги           235 non-null    float64
 7   Ср. выручка с одного клиента, руб      235 non-null    float64
 8   Уникальные клиенты                     235 non-null    int64  
dtypes: float64(4), int64(3), str(2)
memory usage: 16.7 KB


In [24]:
users_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 2711 entries, 0 to 2710
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_user          2711 non-null   int64  
 1   Имя              2711 non-null   str    
 2   Телефон          2711 non-null   str    
 3   Категории        2711 non-null   str    
 4   Оплачено в руб   2711 non-null   float64
 5   Последний визит  2711 non-null   str    
dtypes: float64(1), int64(1), str(4)
memory usage: 127.2 KB


In [25]:
user_item_year_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 28760 entries, 0 to 28759
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Имя мастера            28760 non-null  str    
 1   Специализация мастера  28760 non-null  str    
 2   Имя клиента            28760 non-null  str    
 3   Время визита           28760 non-null  str    
 4   Категория услуги       28760 non-null  str    
 5   Название услуги        28760 non-null  str    
 6   Стоимость, руб         28760 non-null  float64
 7   Категория              28760 non-null  str    
 8   id_user                28760 non-null  int64  
 9   id_item                28760 non-null  int64  
dtypes: float64(1), int64(2), str(7)
memory usage: 2.2 MB


Выгрузка маппера

In [12]:
sys.path.append(str(Path.cwd().parent.parent))
from utils import IDMapper

In [28]:
mapper_path = Path('../../../results/mappers/id_mappers.pkl')
with open(mapper_path, "rb") as f:
    mappers = pickle.load(f)

user_mapper = mappers['user_mapper']
item_mapper = mappers['item_mapper']

Выгрузка IALS модели

In [29]:
model_path = Path('../../../results/models/IALS_model.pkl')
with open(model_path, "rb") as f:
    model = pickle.load(f)